# Data Vortex — Phase 2: Relational Database Setup & Validation

## 1. Overview & Objectives
This notebook demonstrates the end-to-end relational database setup for **Data Vortex Round 1 Phase 2** using **SQLite 3**.

### Core Principles:
- **Source:** Exclusively ingests from `data/cleaned/Social_Engine_Users_Cleaned.csv` and `data/cleaned/Social_Engine_Posts_Cleaned.csv`.
- **Immutability:** Zero alterations to the cleaned datasets.
- **Relational Integrity:** Strict foreign key enforcement via `PRAGMA foreign_keys = ON;` (`posts.user_id -> users.user_id`).
- **NULL Fidelity:** Missing attributes in `platform`, `text_content`, and `likes` are preserved as SQL `NULL` without imputation.

In [ ]:
import os
import csv
import sqlite3
import pandas as pd

# File Paths (Relative to project root or notebook directory)
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(BASE_DIR, "data", "data_vortex.db")
SCHEMA_PATH = os.path.join(BASE_DIR, "sql", "01_schema.sql")
USERS_CSV = os.path.join(BASE_DIR, "data", "cleaned", "Social_Engine_Users_Cleaned.csv")
POSTS_CSV = os.path.join(BASE_DIR, "data", "cleaned", "Social_Engine_Posts_Cleaned.csv")

print(f"Project Base Directory: {BASE_DIR}")
print(f"Database File:         {DB_PATH}")

## 2. SQLite Connection & DDL Schema Execution
Establish the SQLite database connection, enable foreign key constraints, and apply `sql/01_schema.sql`.

In [ ]:
# Connect to SQLite and apply schema
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
    schema_sql = f.read()

conn.executescript(schema_sql)
print("DDL Schema executed successfully. Tables 'users' and 'posts' created with indexes.")

## 3. Data Ingestion from Cleaned CSVs
Load cleaned records into SQLite. Empty fields and literal `NULL` tokens are mapped to Python `None` to store as SQL `NULL`.

In [ ]:
cursor = conn.cursor()

# 1. Load Users
users_rows = []
with open(USERS_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        users_rows.append((
            row["user_id"].strip(),
            row["location"].strip(),
            row["language"].strip(),
            row["account_created"].strip(),
            int(row["follower_count"].strip())
        ))

cursor.executemany(
    "INSERT INTO users (user_id, location, language, account_created, follower_count) VALUES (?, ?, ?, ?, ?);",
    users_rows
)
conn.commit()
print(f"Loaded {len(users_rows):,} users into 'users' table.")

# 2. Load Posts
posts_rows = []
with open(POSTS_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        post_id = row["post_id"].strip()
        user_id = row["user_id"].strip()
        
        raw_plat = row["platform"].strip() if row["platform"] else ""
        platform = raw_plat if raw_plat != "" else None
        
        raw_text = row["text_content"].strip() if row["text_content"] else ""
        text_content = raw_text if (raw_text != "" and raw_text != "NULL") else None
        
        timestamp = row["timestamp"].strip()
        
        raw_likes = row["likes"].strip() if row["likes"] else ""
        likes = int(raw_likes) if raw_likes != "" else None
        
        shares = int(row["shares"].strip())
        comments = int(row["comments"].strip())
        
        posts_rows.append((post_id, user_id, platform, text_content, timestamp, likes, shares, comments))

cursor.executemany(
    "INSERT INTO posts (post_id, user_id, platform, text_content, timestamp, likes, shares, comments) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
    posts_rows
)
conn.commit()
print(f"Loaded {len(posts_rows):,} posts into 'posts' table.")

## 4. Row Count & Primary Key Validation
Verify exact row counts and 100% primary key uniqueness.

In [ ]:
df_counts = pd.read_sql("""
SELECT 
    'users' AS table_name, 
    COUNT(*) AS row_count, 
    COUNT(DISTINCT user_id) AS unique_primary_keys
FROM users
UNION ALL
SELECT 
    'posts' AS table_name, 
    COUNT(*) AS row_count, 
    COUNT(DISTINCT post_id) AS unique_primary_keys
FROM posts;
""", conn)
df_counts

## 5. Foreign Key Integrity & Orphan Record Check
Verify that all post foreign keys resolve to valid user primary keys.

In [ ]:
# PRAGMA foreign key check
fk_check = pd.read_sql("PRAGMA foreign_key_check;", conn)
print(f"Foreign Key Violations (PRAGMA): {len(fk_check)} rows")

# Explicit orphan query
df_orphans = pd.read_sql("""
SELECT COUNT(*) AS orphan_posts
FROM posts
WHERE user_id NOT IN (SELECT user_id FROM users);
""", conn)
print(f"Orphan Posts Count: {df_orphans.iloc[0,0]}")

# Universal user participation
df_participation = pd.read_sql("""
SELECT COUNT(DISTINCT user_id) AS participating_users
FROM posts;
""", conn)
print(f"Participating Users in Posts: {df_participation.iloc[0,0]} (out of 1,500)")

## 6. NULL Handling & Missing Value Verification
Verify that missing attributes in `platform`, `text_content`, and `likes` match the cleaned CSV ground truth.

In [ ]:
df_nulls = pd.read_sql("""
SELECT 
    SUM(CASE WHEN platform IS NULL THEN 1 ELSE 0 END) AS null_platform,
    SUM(CASE WHEN text_content IS NULL THEN 1 ELSE 0 END) AS null_text,
    SUM(CASE WHEN likes IS NULL THEN 1 ELSE 0 END) AS null_likes,
    SUM(CASE WHEN shares IS NULL THEN 1 ELSE 0 END) AS null_shares,
    SUM(CASE WHEN comments IS NULL THEN 1 ELSE 0 END) AS null_comments
FROM posts;
""", conn)
df_nulls

## 7. Basic Sample Analytical Queries
Demonstrate sample relational `JOIN` operations, platform grouping, and engagement aggregations.

In [ ]:
# Sample Query 1: Platform-level Post Volume & Average Interactions
df_platform = pd.read_sql("""
SELECT 
    COALESCE(platform, '[Missing]') AS platform_name,
    COUNT(*) AS total_posts,
    ROUND(AVG(likes), 2) AS avg_likes,
    ROUND(AVG(shares), 2) AS avg_shares,
    ROUND(AVG(comments), 2) AS avg_comments
FROM posts
GROUP BY COALESCE(platform, '[Missing]')
ORDER BY total_posts DESC;
""", conn)
df_platform

In [ ]:
# Sample Query 2: Top 5 Active Users with User Demographic Join
df_top_users = pd.read_sql("""
SELECT 
    u.user_id,
    u.location,
    u.language,
    u.follower_count,
    COUNT(p.post_id) AS total_posts,
    ROUND(AVG(p.likes), 1) AS avg_likes
FROM users u
JOIN posts p ON u.user_id = p.user_id
GROUP BY u.user_id, u.location, u.language, u.follower_count
ORDER BY total_posts DESC
LIMIT 5;
""", conn)
df_top_users

In [ ]:
# Clean up connection
conn.close()
print("SQLite connection closed cleanly.")

## 8. Conclusion
The SQLite database `data/data_vortex.db` is successfully initialized, populated, and validated. It is completely ready for analytical SQL query execution.